# Práctica final · Agente RAG sobre informes 10-K

Versión limpia de entrega.

**Arquitectura final:** Gemini 3.8 Flash + cuatro tools contractuales + XBRL para
hechos estructurados + retrieval híbrido dense/BM25 con filtros + salida
estructurada + guardrail numérico `after_model`.

El notebook también conserva un baseline reproducible, el golden set de 20
preguntas, los tres evaluadores, la ablación de retrieval y la comparación
baseline–final. `Run all` reutiliza resultados existentes o los regenera.

## 1. Setup

In [1]:
# Preparación automática del proyecto en Google Colab.
import os
import subprocess
import sys
import shutil
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    URL_REPO = "https://github.com/rodovilllapa210/https---github.com-PRACTICA-AGENTE-RAG.git"
    CARPETA_REPO = Path("/content/MIAX_2026/Practica_Agente_RAG")

    if not (CARPETA_REPO / ".git").is_dir():
        CARPETA_REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
             URL_REPO, str(CARPETA_REPO)],
            check=True,
        )

    # Si alguno de estos ficheros se subió manualmente al panel de Colab,
    # se copia al repo. El golden oficial NO es dependencia de la entrega.
    OPCIONALES_SUBIDOS = (
        "miax_s2.py", "golden_set.jsonl",
    )
    for nombre in OPCIONALES_SUBIDOS:
        destino = CARPETA_REPO / nombre
        subido = Path("/content") / nombre
        if not destino.is_file() and subido.is_file():
            shutil.copy2(subido, destino)

    NECESARIOS = (
        "miax_s1.py", "miax_s2.py", "corpus_miax_2026.zip",
        "indice_faiss.zip", "golden_set.jsonl",
    )
    faltan = [n for n in NECESARIOS if not (CARPETA_REPO / n).is_file()]
    if faltan:
        raise FileNotFoundError(
            "Faltan archivos necesarios en el repositorio: " + ", ".join(faltan)
        )

    os.chdir(CARPETA_REPO)
    if str(CARPETA_REPO) not in sys.path:
        sys.path.insert(0, str(CARPETA_REPO))
    print("Proyecto preparado en", CARPETA_REPO)
else:
    print("Ejecución local: se usan los archivos de la carpeta actual.")


Proyecto preparado en /content/MIAX_2026/Practica_Agente_RAG


In [2]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Tarda alrededor de minuto y medio: mientras corre, leed la celda siguiente.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-google-genai==4.3.7 google-genai==2.10.0 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2 google-auth==2.49.0
print("Instalación terminada.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 645.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 2.10.0 which is incompatible.
Instalación terminada.


In [3]:
# Clave de Gemini: secreto de Colab o variable de entorno. Nunca se imprime.
import os

HAY_CLAVE = bool(os.environ.get("GEMINI_API_KEY"))

if not HAY_CLAVE:
    try:
        from google.colab import userdata
        clave = userdata.get("GEMINI_API_KEY")
        if clave:
            os.environ["GEMINI_API_KEY"] = clave
            HAY_CLAVE = True
    except Exception:
        pass

print("GEMINI_API_KEY:", "disponible" if HAY_CLAVE else "no disponible")
if not HAY_CLAVE:
    print("Las celdas de datos y métricas funcionan; responder/evaluar requieren clave.")


GEMINI_API_KEY: disponible


In [4]:
# Preparación y verificación del corpus e índice.
import hashlib, pathlib, zipfile

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
URL_RESPALDO = ""          # vacio si no estan alojados
DESTINO = pathlib.Path("corpus")

CANDIDATOS = [
    pathlib.Path("."),
    pathlib.Path("/content"),
    pathlib.Path("/content/drive/MyDrive/MIAX_2026"),
    pathlib.Path("/content/drive/Shareddrives/MIAX_2026"),
]


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


def _localizar(nombre):
    for base in CANDIDATOS:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    if URL_RESPALDO:
        import urllib.request
        destino = pathlib.Path(nombre)
        urllib.request.urlretrieve(f"{URL_RESPALDO}/{nombre}", destino)
        return destino
    return None


try:
    for nombre, esperado in PAQUETES:
        origen = _localizar(nombre)
        assert origen is not None, (
            f"No encuentro {nombre}. Subelo con el panel de ficheros de "
            f"Colab (icono de carpeta a la izquierda), o monta el Drive "
            f"donde este. Buscado en: {[str(c) for c in CANDIDATOS]}"
        )
        obtenido = _sha256(origen)
        assert obtenido == esperado, (
            f"{nombre} no coincide con lo esperado: el fichero esta "
            f"corrupto o es de otra version.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}"
        )
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)

    # Los dos manifiestos declaran el hash de chunks.jsonl. El indice se
    # construyo sobre ESE fichero: si no cuadra, el indice y sus metadatos
    # estan desalineados y el retrieval devuelve el texto equivocado sin
    # dar ningun error.
    huella = _sha256(DESTINO / "chunks.jsonl")
    for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
        ruta = DESTINO / manifiesto
        if ruta.exists():
            assert huella in ruta.read_text(encoding="utf-8"), (
                f"chunks.jsonl no cuadra con {manifiesto}: el indice se "
                "construyo sobre otros fragmentos."
            )

    print("Corpus e indice verificados en", DESTINO.resolve())
    for p in sorted(DESTINO.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(DESTINO))
            print(f"  {rel:28s} {p.stat().st_size / 1e6:7.2f} MB")

except Exception as e:
    print("No se pudo preparar el corpus:", e)
    print("Pide los ficheros al profesor y dejalos junto al notebook.")


Corpus e indice verificados en /content/MIAX_2026/Practica_Agente_RAG/corpus
  LEEME.md                        0.00 MB
  MANIFEST.md                     0.00 MB
  chunks.jsonl                    3.80 MB
  indice/MANIFEST.md              0.00 MB
  indice/chunks_meta.parquet      1.48 MB
  indice/corpus.faiss             2.69 MB
  secciones.jsonl                 3.21 MB
  xbrl_facts.parquet              0.01 MB


In [5]:
import json
import pandas as pd
from pathlib import Path

secciones = pd.DataFrame([
    json.loads(x) for x in Path("corpus/secciones.jsonl").read_text(
        encoding="utf-8"
    ).splitlines() if x.strip()
])
xbrl = pd.read_parquet("corpus/xbrl_facts.parquet")

print(
    f"{len(secciones)} secciones · {len(xbrl)} hechos XBRL · "
    f"{secciones.ticker.nunique()} compañías"
)

48 secciones · 135 hechos XBRL · 6 compañías


In [6]:
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.8-flash"

modelo = None
if HAY_CLAVE:
    try:
        # Gemini 3.8 Flash: dejamos el muestreo en la configuración
        # recomendada por el proveedor; no fijamos temperature/top_p/top_k.
        modelo = init_chat_model(MODELO)
        print("Modelo preparado:", MODELO)
        print("Sampling: model_default")
    except Exception as e:
        print(f"No se pudo crear el modelo ({type(e).__name__}: {e}).")

import pathlib
assert pathlib.Path("corpus/chunks.jsonl").is_file(), \
    "El corpus no está preparado."
assert pathlib.Path("corpus/indice/corpus.faiss").is_file(), \
    "Falta el índice FAISS."
print("§1 listo.")


Modelo preparado: google_genai:gemini-3.8-flash
Sampling: model_default
§1 listo.


## 2. Herramientas y retrieval final

In [7]:
import numpy as np
import miax_s2
from langchain.tools import tool

INDICE, META, _ = miax_s2.cargar_indice()
BM25, CHUNKS_BM25 = miax_s2.montar_bm25()
KK_RRF = 60


def _buscar_hibrido(
    query: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> list[dict]:
    """Dense + BM25 con filtros y Reciprocal Rank Fusion."""
    scores, pos = INDICE.search(miax_s2.codificar([query]), INDICE.ntotal)

    densos = []
    for score, idx in zip(scores[0], pos[0]):
        if int(idx) < 0:
            continue
        fila = META.iloc[int(idx)]
        if ticker is not None and fila["ticker"] != ticker:
            continue
        if fiscal_year is not None and int(fila["fiscal_year"]) != int(fiscal_year):
            continue
        if item is not None and fila["item"] != item:
            continue
        densos.append(miax_s2.fila_a_fragmento(fila, score))

    if not densos:
        return []

    rank_dense = {f["chunk_id"]: i for i, f in enumerate(densos, 1)}
    permitidos = set(rank_dense)

    bm_scores = BM25.get_scores(miax_s2.tokenizar(query))
    rank_bm25, p = {}, 0
    for idx in np.argsort(-bm_scores):
        cid = CHUNKS_BM25[int(idx)]["chunk_id"]
        if cid in permitidos:
            p += 1
            rank_bm25[cid] = p

    rrf = {
        cid: 1 / (KK_RRF + rd) + 1 / (KK_RRF + rank_bm25.get(cid, 10**6))
        for cid, rd in rank_dense.items()
    }
    por_id = {f["chunk_id"]: f for f in densos}

    salida = []
    for cid in sorted(rrf, key=rrf.get, reverse=True)[:int(k)]:
        f = dict(por_id[cid])
        f["puntuacion"] = round(float(rrf[cid]), 6)
        salida.append(f)
    return salida


def _formatear(fragmentos: list[dict]) -> str:
    if not fragmentos:
        return "Sin resultados para esa consulta con esos filtros."
    return "\n\n---\n\n".join(
        f"[{f['chunk_id']}] {f['ticker']} FY{f['fiscal_year']} "
        f"Item {f['item']} (RRF {f['puntuacion']:.6f})\n{f['texto']}"
        for f in fragmentos
    )


@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve el valor exacto de una magnitud financiera reportada en XBRL.

    Args:
        ticker: Símbolo bursátil.
        fiscal_year: Ejercicio fiscal.
        concept: Concepto US-GAAP.
    """
    filas = xbrl[
        (xbrl.ticker == ticker)
        & (xbrl.fiscal_year == int(fiscal_year))
        & (xbrl.concept == concept)
    ]
    if filas.empty:
        disponibles = sorted(
            xbrl[
                (xbrl.ticker == ticker)
                & (xbrl.fiscal_year == int(fiscal_year))
            ].concept.unique()
        )
        return (
            f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
            f"Conceptos disponibles: {', '.join(disponibles) or 'ninguno'}."
        )
    f = filas.iloc[0]
    return (
        f"{ticker} FY{fiscal_year} {concept} = {f.value:,.0f} {f.unit} "
        f"(cierre {f.period_end}, {f.form})"
    )


@tool
def search_filings(
    query: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> str:
    """Busca texto relevante en los 10-K mediante dense + BM25/RRF.

    Args:
        query: Consulta, preferiblemente en inglés.
        ticker: Filtro opcional por compañía.
        fiscal_year: Filtro opcional por ejercicio.
        item: Filtro opcional: '1A', '7', '7A' u '8'.
        k: Número de fragmentos.
    """
    return _formatear(_buscar_hibrido(query, ticker, fiscal_year, item, k))


@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    """Devuelve el texto completo de una sección; úsala solo como fallback."""
    filas = secciones[
        (secciones.ticker == ticker)
        & (secciones.fiscal_year == int(fiscal_year))
        & (secciones.item == item)
    ]
    if filas.empty:
        return f"No hay Item {item} de {ticker} FY{fiscal_year}."
    return filas.iloc[0].texto


@tool
def list_available() -> str:
    """Lista compañías, ejercicios y secciones disponibles en el corpus."""
    lineas = []
    for ticker, filas in secciones.groupby("ticker", sort=True):
        years = sorted(filas.fiscal_year.astype(int).unique())
        items = sorted(filas.item.astype(str).unique())
        lineas.append(f"{ticker}: FY{years} · Items {items}")
    return "\n".join(lineas)


HERRAMIENTAS = [
    list_available, get_xbrl_fact, search_filings, read_section
]
NOMBRES_HERRAMIENTAS = {t.name for t in HERRAMIENTAS}
print("Herramientas:", ", ".join(t.name for t in HERRAMIENTAS))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Herramientas: list_available, get_xbrl_fact, search_filings, read_section


## 3. Agente final y baseline

In [8]:
from typing import Literal
from pydantic import BaseModel, Field, model_validator
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentState, ToolCallLimitMiddleware, after_model
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

SYSTEM = """Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

- Magnitudes contables estandarizadas disponibles en XBRL:
  usa get_xbrl_fact.
- Riesgos, estrategia, guidance, porcentajes narrativos y explicaciones:
  usa search_filings.
- Una pregunta puede combinar XBRL y texto.
- El corpus está en inglés: formula las búsquedas en inglés.
- Si el dato no está en el corpus, dilo; no lo estimes.
- Si la pregunta es solo XBRL, no confirmes la cifra con texto.
- Usa search_filings con k=5 por defecto y deja de buscar cuando haya
  evidencia suficiente; read_section es último recurso.
- En comparativas, `cifra` es siempre el valor de la magnitud principal en
  el ejercicio final; diferencias y porcentajes van en `respuesta`.
- Si `fuente` es "texto" o "ambas", `cita` y `chunk_id` son obligatorios y
  deben corresponder al mismo fragmento recuperado.
"""


class RespuestaFinanciera(BaseModel):
    respuesta: str
    cifra: float | None = None
    unidad: str | None = None
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"]
    cita: str | None = None
    chunk_id: str | None = None

    @model_validator(mode="after")
    def trazabilidad(self):
        if self.fuente in {"texto", "ambas"}:
            if not self.cita or not self.chunk_id:
                raise ValueError(
                    "fuente texto/ambas exige cita y chunk_id."
                )
        return self


MARCA_GUARDRAIL = "VERIFICACIÓN AUTOMÁTICA"
TOLERANCIA_GUARDRAIL = 0.01


def _contenido(m) -> str:
    return str(m.get("content", "")) if isinstance(m, dict) else str(
        getattr(m, "content", "")
    )


@after_model(can_jump_to=["model"])
def verificar_cifras_contra_xbrl(
    state: AgentState, runtime: Runtime
) -> dict | None:
    """Contrasta la cifra estructurada con XBRL y permite una corrección."""
    r = state.get("structured_response")
    if (
        r is None
        or getattr(r, "cifra", None) is None
        or getattr(r, "fuente", None) not in {"xbrl", "ambas"}
    ):
        return None

    ticker, ejercicio = getattr(r, "ticker", None), getattr(r, "ejercicio", None)
    if ticker is None or ejercicio is None:
        return None

    # Una sola corrección evita un bucle verifier -> model.
    if any(MARCA_GUARDRAIL in _contenido(m) for m in state.get("messages", [])):
        return None

    hechos = xbrl[
        (xbrl.ticker == ticker)
        & (xbrl.fiscal_year == int(ejercicio))
    ]
    if hechos.empty:
        return None

    afirmada = float(r.cifra)
    if any(
        miax_s2.cuadra(afirmada, float(v), TOLERANCIA_GUARDRAIL)
        for v in hechos.value
    ):
        return None

    disponibles = "; ".join(
        f"{f.concept}={float(f.value):,.0f} {f.unit}"
        for f in hechos.itertuples()
    )
    return {
        "messages": [{
            "role": "user",
            "content": (
                f"{MARCA_GUARDRAIL}: {afirmada:,.6g} no coincide con XBRL "
                f"de {ticker} FY{ejercicio} (tolerancia 1%). "
                f"Valores disponibles: {disponibles}. Corrige la respuesta."
            ),
        }],
        "jump_to": "model",
    }


limite_tools = ToolCallLimitMiddleware(run_limit=11, exit_behavior="end")

agente = None
if HAY_CLAVE:
    agente = create_agent(
        model=modelo,
        tools=HERRAMIENTAS,
        system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        checkpointer=InMemorySaver(),
        middleware=[limite_tools, verificar_cifras_contra_xbrl],
    )
    print("Agente final preparado.")

Agente final preparado.


In [9]:
import miax_s1

def _search_baseline(
    query: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> str:
    """Busca con el retriever denso original del baseline."""
    return miax_s1.formatear_fragmentos(
        miax_s1.buscar(
            query,
            ticker=ticker,
            fiscal_year=fiscal_year,
            item=item,
            k=k,
        )
    )

search_filings_baseline = tool("search_filings")(_search_baseline)
HERRAMIENTAS_BASELINE = [
    list_available, get_xbrl_fact, search_filings_baseline, read_section
]

SYSTEM_BASELINE = """Responde sobre los 10-K usando solo las herramientas.
Para cualquier cifra usa get_xbrl_fact; para riesgos, estrategia o comentarios
usa search_filings. Busca en inglés, cita el chunk_id y no estimes datos que no
estén en el corpus."""

agente_baseline = None
if HAY_CLAVE:
    agente_baseline = create_agent(
        model=modelo,
        tools=HERRAMIENTAS_BASELINE,
        system_prompt=SYSTEM_BASELINE,
        response_format=RespuestaFinanciera,
        checkpointer=InMemorySaver(),
        middleware=[
            ToolCallLimitMiddleware(run_limit=11, exit_behavior="end"),
            verificar_cifras_contra_xbrl,
        ],
    )
    print("Baseline preparado.")

Baseline preparado.


## 4. Golden set y mejoras de retrieval

In [10]:
RUTA_GOLDEN = Path("golden_set.jsonl")
if not RUTA_GOLDEN.is_file():
    raise FileNotFoundError("Falta golden_set.jsonl.")

def leer_jsonl(ruta):
    return [
        json.loads(x) for x in Path(ruta).read_text(
            encoding="utf-8"
        ).splitlines() if x.strip()
    ]

golden = leer_jsonl(RUTA_GOLDEN)

CAMPOS = {
    "id", "pregunta", "familia", "ticker", "fiscal_year",
    "respuesta_esperada", "cifra_esperada", "unidad", "concept_xbrl",
    "item_esperado", "ancla_texto", "ancla_inicio", "ancla_fin",
    "chunk_id_esperado", "herramienta_esperada", "autor",
}

def validar(preguntas, exigir_20=True):
    problemas, vistos = [], set()
    tickers = set(secciones.ticker)

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        faltan = CAMPOS - set(p)
        if faltan:
            problemas.append(f"{pid}: faltan {sorted(faltan)}")
            continue
        if pid in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(pid)
        if p["familia"] not in {"numerica", "extractiva", "comparativa"}:
            problemas.append(f"{pid}: familia inválida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: ticker fuera del corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p["cifra_esperada"] is None:
                problemas.append(f"{pid}: falta cifra_esperada")
        if p["familia"] in {"extractiva", "comparativa"}:
            if not p["ancla_texto"]:
                problemas.append(f"{pid}: falta ancla_texto")
        if not p["herramienta_esperada"]:
            problemas.append(f"{pid}: falta herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(f"se exigen 20 preguntas; hay {len(preguntas)}")
        if sum(p["familia"] == "comparativa" for p in preguntas) < 6:
            problemas.append("se exigen al menos 6 comparativas")
    return problemas

problemas = validar(golden)
assert not problemas, "\n".join(problemas)
print(
    f"Golden OK: {len(golden)} preguntas · "
    f"{sum(g['familia']=='comparativa' for g in golden)} comparativas"
)

Golden OK: 20 preguntas · 6 comparativas


In [11]:
# Ablación realizada durante el desarrollo sobre el benchmark oficial
# de S2 (13 preguntas con ancla). La función inferior permite repetirla
# sobre nuestro golden_set.jsonl.
# Query rewriting fue medido, pero NO se incorporó al agente final porque
# el propio agente ya formula consultas en inglés y una segunda llamada añade
# coste y puede introducir regresiones.
ablacion_retrieval = pd.DataFrame([
    ("Denso plano",              0.308),
    ("+ metadatos",              0.462),
    ("+ BM25/RRF",               0.538),
    ("+ query rewriting",        0.692),
    ("Rewriting + BM25/RRF",     0.769),
], columns=["configuración", "recall@5"])

display(ablacion_retrieval.style.format({"recall@5": "{:.1%}"}))


def reproducir_ablacion(preguntas=golden):
    """Reproduce las cinco configuraciones; rewriting usa Gemini una vez/caso."""
    casos = [g for g in preguntas if g.get("ancla_texto")]
    reescritas = {}

    def dense(q, g, filtros=True):
        return miax_s1.buscar(
            q,
            ticker=g["ticker"] if filtros else None,
            fiscal_year=g["fiscal_year"] if filtros else None,
            item=g["item_esperado"] if filtros else None,
            k=5,
        )

    def rewrite(g):
        if g["id"] not in reescritas:
            m = modelo.invoke(
                "Reescribe SOLO como consulta de búsqueda en inglés para un "
                "10-K, usando terminología financiera:\n" + g["pregunta"]
            )
            reescritas[g["id"]] = m.text.strip()
        return reescritas[g["id"]]

    configs = {
        "Denso plano": lambda g: dense(g["pregunta"], g, False),
        "+ metadatos": lambda g: dense(g["pregunta"], g, True),
        "+ BM25/RRF": lambda g: _buscar_hibrido(
            g["pregunta"], g["ticker"], g["fiscal_year"], g["item_esperado"], 5
        ),
        "+ query rewriting": lambda g: dense(rewrite(g), g, True),
        "Rewriting + BM25/RRF": lambda g: _buscar_hibrido(
            rewrite(g), g["ticker"], g["fiscal_year"], g["item_esperado"], 5
        ),
    }

    filas = []
    for nombre, buscar in configs.items():
        hit = sum(miax_s2.acierta(g, buscar(g)) for g in casos)
        filas.append((nombre, hit / len(casos)))
    return pd.DataFrame(filas, columns=["configuración", "recall@5"])

print("Para repetir la ablación: reproducir_ablacion()")

,configuración,recall@5
0,Denso plano,30.8%
1,+ metadatos,46.2%
2,+ BM25/RRF,53.8%
3,+ query rewriting,69.2%
4,Rewriting + BM25/RRF,76.9%


Para repetir la ablación: reproducir_ablacion()


## 5. Evaluación y resultados

In [12]:
import time
import uuid

PRECIO_INPUT_USD_M = 0.75
PRECIO_OUTPUT_USD_M = 3.75

CHUNKS = [
    json.loads(x) for x in Path("corpus/chunks.jsonl").read_text(
        encoding="utf-8"
    ).splitlines() if x.strip()
]
POR_ID = {c["chunk_id"]: c for c in CHUNKS}


def responder(pregunta: str):
    if agente is None:
        raise RuntimeError("Configura GEMINI_API_KEY.")
    return agente.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": f"eval-{uuid.uuid4()}"}},
    )


def responder_baseline(pregunta: str):
    if agente_baseline is None:
        raise RuntimeError("Configura GEMINI_API_KEY.")
    return agente_baseline.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": f"base-{uuid.uuid4()}"}},
    )


def cita_correcta(item, resultado):
    r = resultado.get("structured_response")
    if r is None:
        return None if item["familia"] == "numerica" else False
    if not r.chunk_id:
        return None if item["familia"] == "numerica" else False
    if r.chunk_id not in POR_ID or not r.cita:
        return False
    return (
        miax_s2.normalizar(str(r.cita))[:120]
        in miax_s2.normalizar(POR_ID[r.chunk_id]["texto"])
    )


def cifra_coincide_xbrl(item, resultado):
    esperada = item.get("cifra_esperada")
    if esperada is None:
        return None
    r = resultado.get("structured_response")
    return bool(
        r is not None and r.cifra is not None
        and miax_s2.cuadra(float(r.cifra), float(esperada), 0.01)
    )


def uso_la_tool_correcta(item, resultado):
    usadas = set(miax_s2.herramientas_usadas(resultado))
    return set(item["herramienta_esperada"]).issubset(usadas)


def _recall(item, final=True):
    if not item.get("ancla_texto"):
        return None
    if final:
        rec = _buscar_hibrido(
            item["pregunta"], item["ticker"], item["fiscal_year"],
            item["item_esperado"], 5
        )
    else:
        rec = miax_s1.buscar(
            item["pregunta"], ticker=item["ticker"],
            fiscal_year=item["fiscal_year"], item=item["item_esperado"], k=5
        )
    return miax_s2.acierta(item, rec)


def _coste(resultado):
    entrada, salida = miax_s2.tokens_de(resultado)
    return (
        entrada * PRECIO_INPUT_USD_M + salida * PRECIO_OUTPUT_USD_M
    ) / 1e6


def _respuesta_dict(resultado):
    r = resultado.get("structured_response")
    return r.model_dump() if hasattr(r, "model_dump") else r


def _trayectoria(resultado):
    salidas = {
        getattr(m, "tool_call_id", None): str(getattr(m, "content", ""))
        for m in resultado.get("messages", [])
        if getattr(m, "tool_call_id", None)
    }
    out = []
    for m in resultado.get("messages", []):
        for c in getattr(m, "tool_calls", []) or []:
            if c.get("name") in NOMBRES_HERRAMIENTAS:
                out.append({
                    "name": c["name"],
                    "args": c.get("args", {}),
                    "output": salidas.get(c.get("id")),
                })
    return out


def _evaluar_una(caso, respondedor, final):
    t0 = time.perf_counter()
    error = None
    try:
        resultado = respondedor(caso["pregunta"])
    except Exception as exc:
        resultado = {"messages": [], "structured_response": None}
        error = f"{type(exc).__name__}: {exc}"
    latencia = time.perf_counter() - t0

    ev = {
        "cita": cita_correcta(caso, resultado),
        "cifra": cifra_coincide_xbrl(caso, resultado),
        "trayectoria": uso_la_tool_correcta(caso, resultado),
        "recall@5": _recall(caso, final=final),
    }
    aplicables = [v for k, v in ev.items() if k != "recall@5" and v is not None]

    return {
        "id": caso["id"],
        "familia": caso["familia"],
        "pregunta": caso["pregunta"],
        "respuesta": _respuesta_dict(resultado),
        "trayectoria": _trayectoria(resultado),
        "evaluacion": ev,
        "acierto": bool(aplicables) and all(aplicables) and error is None,
        "coste_usd": _coste(resultado),
        "latencia_s": latencia,
        "n_llamadas": len(miax_s2.herramientas_usadas(resultado)),
        "error": error,
    }


def _evaluar(ruta_jsonl, respondedor, final):
    casos = leer_jsonl(ruta_jsonl)
    problemas = validar(casos, exigir_20=False)
    if problemas:
        raise ValueError("\n".join(problemas))

    registros = [_evaluar_una(c, respondedor, final) for c in casos]
    tabla = pd.DataFrame([{
        "id": r["id"],
        "familia": r["familia"],
        "acierto": r["acierto"],
        "cita": r["evaluacion"]["cita"],
        "cifra": r["evaluacion"]["cifra"],
        "trayectoria": r["evaluacion"]["trayectoria"],
        "recall": r["evaluacion"]["recall@5"],
        "coste_usd": r["coste_usd"],
        "latencia_s": r["latencia_s"],
        "llamadas": r["n_llamadas"],
        "error": r["error"],
    } for r in registros])
    return registros, tabla


def evaluar(ruta_jsonl):
    """Interfaz pública para golden y holdout."""
    return _evaluar(ruta_jsonl, responder, final=True)


def evaluar_baseline(ruta_jsonl):
    return _evaluar(ruta_jsonl, responder_baseline, final=False)


def ejecutar_y_guardar(ruta, prefijo, funcion):
    carpeta = Path("resultados")
    carpeta.mkdir(exist_ok=True)
    p_json = carpeta / f"{prefijo}_registros.jsonl"
    p_csv = carpeta / f"{prefijo}_tabla.csv"

    if p_json.is_file() and p_csv.is_file():
        registros = leer_jsonl(p_json)
        tabla = pd.read_csv(p_csv)
        if "acierto" not in tabla.columns and "acierto_oficial" in tabla.columns:
            tabla = tabla.rename(columns={"acierto_oficial": "acierto"})
        print(prefijo, ": resultados existentes cargados.")
        return registros, tabla

    registros, tabla = funcion(ruta)
    with p_json.open("w", encoding="utf-8") as f:
        for r in registros:
            f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    tabla.to_csv(p_csv, index=False)
    print(prefijo, ": resultados regenerados.")
    return registros, tabla

## 6. Holdout del profesor · evaluación posterior al congelado

El agente, las tools, el prompt, el retrieval y los parámetros son exactamente
los del notebook final congelado. **No se modifica el sistema después de conocer
el holdout.**

Este bloque:

1. carga y verifica las 10 preguntas entregadas por el profesor;
2. adapta únicamente la **evaluación** a dos casos negativos del holdout
   (`respuesta_en_corpus=False`);
3. ejecuta nuestro agente final sobre las 10 preguntas;
4. genera el detalle pregunta a pregunta;
5. compara el resultado del holdout con la ejecución final previa sobre nuestro
   golden de 20 preguntas.

La comparación con nuestro golden utiliza los resultados de la ejecución final
ya cerrada: **20/20, recall@5 61,5 %, coste medio $0,0122, latencia 8,73 s y
4,05 llamadas/pregunta**. Por defecto no se vuelven a gastar 20 llamadas
adicionales para repetir esa corrida.

In [13]:
# Holdout oficial recibido. Verificamos su CONTENIDO, no los bytes de salto de línea.
# El fichero original usa CRLF y el fallback del notebook usa LF; ambos representan
# exactamente los mismos 10 registros JSONL.
HOLDOUT_CANONICAL_SHA256 = "612fd7d94c618289977ab979cc7a2bab5d639cc406f6b59d413b60aa31dc7b68"
RUTA_HOLDOUT = Path("holdout.jsonl")

# Si se subió al panel de Colab después de clonar el repo, lo copiamos.
if not RUTA_HOLDOUT.is_file():
    candidato = Path("/content/holdout.jsonl")
    if candidato.is_file():
        shutil.copy2(candidato, RUTA_HOLDOUT)

# Fallback autocontenido con exactamente el contenido lógico recibido.
if not RUTA_HOLDOUT.is_file():
    RUTA_HOLDOUT.write_text("{\"id\": \"ho-001\", \"pregunta\": \"¿Qué dice NVIDIA en su 10-K de FY2024 sobre las garantías de suministro de obleas y componentes?\", \"familia\": \"extractiva\", \"ticker\": \"NVDA\", \"fiscal_year\": 2024, \"respuesta_esperada\": \"Que no tiene garantizado el suministro de obleas, componentes ni capacidad, y que sus entregas y producción pueden no ser lineales dentro de un trimestre o un año.\", \"cifra_esperada\": null, \"unidad\": null, \"concept_xbrl\": null, \"item_esperado\": \"1A\", \"ancla_texto\": \"We are not provided guaranteed wafer, component and capacity supply, and our supply deliveries and production may be non-linear within a quarter or year.\", \"ancla_inicio\": 9979, \"ancla_fin\": 10132, \"chunk_id_esperado\": \"NVDA-2024-1A-0005\", \"herramienta_esperada\": [\"search_filings\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"texto\"}\n{\"id\": \"ho-002\", \"pregunta\": \"¿Cuáles son las principales exposiciones a divisas de Alphabet según el apartado de riesgo de mercado de su 10-K de 2025?\", \"familia\": \"extractiva\", \"ticker\": \"GOOGL\", \"fiscal_year\": 2025, \"respuesta_esperada\": \"El dólar australiano, la libra esterlina, el dólar canadiense, el euro y el yen japonés.\", \"cifra_esperada\": null, \"unidad\": null, \"concept_xbrl\": null, \"item_esperado\": \"7A\", \"ancla_texto\": \"Principal currency exposures include the Australian dollar, British pound, Canadian dollar, Euro, and Japanese yen.\", \"ancla_inicio\": 500, \"ancla_fin\": 615, \"chunk_id_esperado\": \"GOOGL-2025-7A-0000\", \"herramienta_esperada\": [\"search_filings\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"texto\"}\n{\"id\": \"ho-003\", \"pregunta\": \"¿Qué ocurrió con las ventas de Apple en la Gran China en 2025 y a qué se debió?\", \"familia\": \"extractiva\", \"ticker\": \"AAPL\", \"fiscal_year\": 2025, \"respuesta_esperada\": \"Bajaron respecto a 2024, sobre todo por menores ventas de iPhone, compensadas en parte por mayores ventas de Mac.\", \"cifra_esperada\": null, \"unidad\": null, \"concept_xbrl\": null, \"item_esperado\": \"7\", \"ancla_texto\": \"Greater China\\n\\nGreater China net sales decreased during 2025 compared to 2024 primarily due to lower net sales of iPhone, partially offset by higher net sales of Mac.\", \"ancla_inicio\": 4759, \"ancla_fin\": 4925, \"chunk_id_esperado\": \"AAPL-2025-7-0002\", \"herramienta_esperada\": [\"search_filings\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"texto\"}\n{\"id\": \"ho-004\", \"pregunta\": \"¿Cuál era el activo total de Microsoft al cierre del ejercicio fiscal 2024?\", \"familia\": \"numerica\", \"ticker\": \"MSFT\", \"fiscal_year\": 2024, \"respuesta_esperada\": \"512.163 millones de dólares.\", \"cifra_esperada\": 512163000000.0, \"unidad\": \"USD\", \"concept_xbrl\": \"Assets\", \"item_esperado\": null, \"ancla_texto\": null, \"ancla_inicio\": null, \"ancla_fin\": null, \"chunk_id_esperado\": null, \"herramienta_esperada\": [\"get_xbrl_fact\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"xbrl\"}\n{\"id\": \"ho-005\", \"pregunta\": \"¿Cuál fue el beneficio bruto de Amazon en 2025?\", \"familia\": \"numerica\", \"ticker\": \"AMZN\", \"fiscal_year\": 2025, \"respuesta_esperada\": \"Amazon no etiqueta GrossProfit en us-gaap, así que esa cifra no está en el corpus. La respuesta correcta es decirlo, no estimarla a partir de los ingresos y el coste de ventas.\", \"cifra_esperada\": null, \"unidad\": null, \"concept_xbrl\": \"GrossProfit\", \"item_esperado\": null, \"ancla_texto\": null, \"ancla_inicio\": null, \"ancla_fin\": null, \"chunk_id_esperado\": null, \"herramienta_esperada\": [\"get_xbrl_fact\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": false, \"fuente_esperada\": \"ninguna\"}\n{\"id\": \"ho-006\", \"pregunta\": \"¿Cuál fue el revenue de NVIDIA en el ejercicio fiscal 2023?\", \"familia\": \"numerica\", \"ticker\": \"NVDA\", \"fiscal_year\": 2023, \"respuesta_esperada\": \"El corpus solo cubre FY2024 y FY2025, así que FY2023 no está. La respuesta correcta es decirlo; comprobarlo es para lo que existe list_available.\", \"cifra_esperada\": null, \"unidad\": null, \"concept_xbrl\": \"Revenues\", \"item_esperado\": null, \"ancla_texto\": null, \"ancla_inicio\": null, \"ancla_fin\": null, \"chunk_id_esperado\": null, \"herramienta_esperada\": [\"list_available\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": false, \"fuente_esperada\": \"ninguna\"}\n{\"id\": \"ho-007\", \"pregunta\": \"¿Cuánto crecieron los ingresos de Apple entre FY2024 y FY2025, y qué explica la evolución del iPhone?\", \"familia\": \"comparativa\", \"ticker\": \"AAPL\", \"fiscal_year\": 2025, \"respuesta_esperada\": \"De 391.035 millones de dólares a 416.161 millones de dólares (+6,4 %). Las ventas de iPhone subieron por los modelos Pro.\", \"cifra_esperada\": 416161000000.0, \"unidad\": \"USD\", \"concept_xbrl\": \"RevenueFromContractWithCustomerExcludingAssessedTax\", \"item_esperado\": \"7\", \"ancla_texto\": \"iPhone\\n\\niPhone net sales increased during 2025 compared to 2024 due to higher net sales of Pro models.\", \"ancla_inicio\": 5850, \"ancla_fin\": 5952, \"chunk_id_esperado\": \"AAPL-2025-7-0003\", \"herramienta_esperada\": [\"get_xbrl_fact\", \"search_filings\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"ambas\"}\n{\"id\": \"ho-008\", \"pregunta\": \"¿Cuánto aumentó el gasto en I+D de Alphabet entre 2024 y 2025, y qué advierte la compañía sobre la relación entre el gasto en compensación y la plantilla?\", \"familia\": \"comparativa\", \"ticker\": \"GOOGL\", \"fiscal_year\": 2025, \"respuesta_esperada\": \"De 49.326 millones de dólares a 61.087 millones de dólares (+23,8 %). La compañía advierte de que el gasto en compensación no sigue directamente a la plantilla, entre otras cosas por las acciones que consolidan con el tiempo.\", \"cifra_esperada\": 61087000000.0, \"unidad\": \"USD\", \"concept_xbrl\": \"ResearchAndDevelopmentExpense\", \"item_esperado\": \"7\", \"ancla_texto\": \"Additionally, fluctuations in employee compensation expenses may not directly correlate with changes in headcount, due to factors such as annual SBC awards that vest over time.\", \"ancla_inicio\": 13806, \"ancla_fin\": 13982, \"chunk_id_esperado\": \"GOOGL-2025-7-0006\", \"herramienta_esperada\": [\"get_xbrl_fact\", \"search_filings\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"ambas\"}\n{\"id\": \"ho-009\", \"pregunta\": \"¿Cómo cambió el beneficio bruto de NVIDIA entre FY2024 y FY2025, y qué margen bruto reportó en cada ejercicio?\", \"familia\": \"comparativa\", \"ticker\": \"NVDA\", \"fiscal_year\": 2025, \"respuesta_esperada\": \"De 44.301 millones de dólares a 97.858 millones de dólares (+120,9 %), con el margen bruto subiendo del 72,7 % en FY2024 al 75,0 % en FY2025.\", \"cifra_esperada\": 97858000000.0, \"unidad\": \"USD\", \"concept_xbrl\": \"GrossProfit\", \"item_esperado\": \"7\", \"ancla_texto\": \"Gross margins increased to 75.0% in fiscal year 2025 from 72.7% in fiscal year 2024.\", \"ancla_inicio\": 27404, \"ancla_fin\": 27488, \"chunk_id_esperado\": \"NVDA-2025-7-0015\", \"herramienta_esperada\": [\"get_xbrl_fact\", \"search_filings\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"ambas\"}\n{\"id\": \"ho-010\", \"pregunta\": \"¿Cómo evolucionó el beneficio operativo de Microsoft entre FY2024 y FY2025, y qué dice la dirección sobre sus segmentos?\", \"familia\": \"comparativa\", \"ticker\": \"MSFT\", \"fiscal_year\": 2025, \"respuesta_esperada\": \"De 109.433 millones de dólares a 128.528 millones de dólares (+17,4 %): 19.100 millones más, un 17 %, con crecimiento en todos sus segmentos.\", \"cifra_esperada\": 128528000000.0, \"unidad\": \"USD\", \"concept_xbrl\": \"OperatingIncomeLoss\", \"item_esperado\": \"7\", \"ancla_texto\": \"Operating income increased $19.1 billion or 17% with growth across each of our segments.\", \"ancla_inicio\": 14335, \"ancla_fin\": 14423, \"chunk_id_esperado\": \"MSFT-2025-7-0007\", \"herramienta_esperada\": [\"get_xbrl_fact\", \"search_filings\"], \"autor\": \"holdout\", \"respuesta_en_corpus\": true, \"fuente_esperada\": \"ambas\"}" + "\n", encoding="utf-8")

def _hash_canonico_jsonl(ruta):
    registros = [
        json.loads(line)
        for line in Path(ruta).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    canonico = "\n".join(
        json.dumps(
            r,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )
        for r in registros
    )
    return hashlib.sha256(canonico.encode("utf-8")).hexdigest(), registros

import hashlib

huella_holdout, holdout = _hash_canonico_jsonl(RUTA_HOLDOUT)
assert huella_holdout == HOLDOUT_CANONICAL_SHA256, (
    "El contenido de holdout.jsonl no coincide con el recibido del profesor. "
    f"Esperado: {HOLDOUT_CANONICAL_SHA256} · obtenido: {huella_holdout}"
)

print("Holdout verificado:", len(holdout), "preguntas")
print("SHA-256 canónico:", huella_holdout)

Holdout verificado: 10 preguntas
SHA-256 canónico: 612fd7d94c618289977ab979cc7a2bab5d639cc406f6b59d413b60aa31dc7b68


In [14]:
CAMPOS_HOLDOUT_EXTRA = {"respuesta_en_corpus", "fuente_esperada"}

def validar_holdout_profesor(preguntas):
    """Valida el esquema del holdout sin exigir cifra en casos fuera de corpus."""
    problemas = []
    vistos = set()
    tickers = set(secciones.ticker)

    if len(preguntas) != 10:
        problemas.append(f"se esperaban 10 preguntas; hay {len(preguntas)}")

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        faltan = (CAMPOS | CAMPOS_HOLDOUT_EXTRA) - set(p)
        if faltan:
            problemas.append(f"{pid}: faltan {sorted(faltan)}")
            continue

        if pid in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(pid)

        if p["familia"] not in {"numerica", "extractiva", "comparativa"}:
            problemas.append(f"{pid}: familia inválida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: ticker desconocido")
        if p["fuente_esperada"] not in {"xbrl", "texto", "ambas", "ninguna"}:
            problemas.append(f"{pid}: fuente_esperada inválida")
        if not p["herramienta_esperada"]:
            problemas.append(f"{pid}: falta herramienta_esperada")

        # En el holdout hay dos preguntas deliberadamente fuera de corpus.
        if p["respuesta_en_corpus"]:
            if p["familia"] in {"numerica", "comparativa"}:
                if p["cifra_esperada"] is None:
                    problemas.append(f"{pid}: falta cifra_esperada")
            if p["familia"] in {"extractiva", "comparativa"}:
                if not p["ancla_texto"]:
                    problemas.append(f"{pid}: falta ancla_texto")
        else:
            if p["fuente_esperada"] != "ninguna":
                problemas.append(
                    f"{pid}: fuera de corpus pero fuente_esperada != 'ninguna'"
                )

    return problemas

problemas_holdout = validar_holdout_profesor(holdout)
assert not problemas_holdout, "\n".join(problemas_holdout)

composicion_holdout = pd.DataFrame({
    "n": pd.Series([p["familia"] for p in holdout]).value_counts()
}).rename_axis("familia")

print(
    "Holdout OK ·",
    f"{sum(p['respuesta_en_corpus'] for p in holdout)} en corpus ·",
    f"{sum(not p['respuesta_en_corpus'] for p in holdout)} fuera de corpus"
)
display(composicion_holdout)

Holdout OK · 8 en corpus · 2 fuera de corpus


,n
familia,
comparativa,4
extractiva,3
numerica,3


### Evaluación específica del holdout

No se cambia el agente. Solo ampliamos el evaluador para aprovechar los dos
campos nuevos que trae el fichero del profesor:

- `fuente_esperada`;
- `respuesta_en_corpus`.

Para las preguntas con evidencia textual exigimos además que el `chunk_id`
devuelto contenga el **ancla de evidencia del profesor**, además de que la cita
pertenezca realmente a ese chunk.

Para los dos casos deliberadamente fuera de corpus, el comportamiento correcto
es devolver `fuente="ninguna"` y no inventar una cifra.

In [15]:
def _r_estructurada(resultado):
    return resultado.get("structured_response")


def fuente_correcta_holdout(caso, resultado):
    r = _r_estructurada(resultado)
    return bool(
        r is not None
        and getattr(r, "fuente", None) == caso["fuente_esperada"]
    )


def evidencia_correcta_holdout(caso, resultado):
    """Comprueba cita + chunk y, además, que el chunk contiene el ancla oficial."""
    if not caso.get("ancla_texto"):
        return None

    r = _r_estructurada(resultado)
    if r is None or not getattr(r, "chunk_id", None) or not getattr(r, "cita", None):
        return False
    if r.chunk_id not in POR_ID:
        return False

    texto_chunk = miax_s2.normalizar(POR_ID[r.chunk_id]["texto"])
    cita = miax_s2.normalizar(str(r.cita))
    ancla = miax_s2.normalizar(str(caso["ancla_texto"]))

    # La cita debe proceder del chunk citado y ese chunk debe contener
    # la evidencia oficial. Usamos los primeros 120 caracteres para tolerar
    # diferencias menores de espacios/formato.
    cita_en_chunk = cita[:120] in texto_chunk
    ancla_en_chunk = ancla[:120] in texto_chunk
    return bool(cita_en_chunk and ancla_en_chunk)


def cifra_correcta_holdout(caso, resultado):
    esperada = caso.get("cifra_esperada")
    r = _r_estructurada(resultado)

    if esperada is not None:
        return bool(
            r is not None
            and getattr(r, "cifra", None) is not None
            and miax_s2.cuadra(float(r.cifra), float(esperada), 0.01)
        )

    # Caso negativo del profesor: si la respuesta NO está en el corpus,
    # no debe aparecer una cifra inventada.
    if not caso.get("respuesta_en_corpus", True):
        return bool(
            r is not None
            and getattr(r, "cifra", None) is None
        )

    return None


def trayectoria_correcta_holdout(caso, resultado):
    usadas = set(miax_s2.herramientas_usadas(resultado))
    return set(caso["herramienta_esperada"]).issubset(usadas)


def ausencia_correcta_holdout(caso, resultado):
    if caso.get("respuesta_en_corpus", True):
        return None
    r = _r_estructurada(resultado)
    return bool(
        r is not None
        and getattr(r, "fuente", None) == "ninguna"
        and getattr(r, "cifra", None) is None
    )


def _evaluar_una_holdout(caso):
    t0 = time.perf_counter()
    error = None

    try:
        resultado = responder(caso["pregunta"])
    except Exception as exc:
        resultado = {"messages": [], "structured_response": None}
        error = f"{type(exc).__name__}: {exc}"

    latencia = time.perf_counter() - t0

    ev = {
        "evidencia": evidencia_correcta_holdout(caso, resultado),
        "cifra": cifra_correcta_holdout(caso, resultado),
        "fuente": fuente_correcta_holdout(caso, resultado),
        "trayectoria": trayectoria_correcta_holdout(caso, resultado),
        "ausencia": ausencia_correcta_holdout(caso, resultado),
        "recall@5": _recall(caso, final=True),
    }

    # recall@5 es una métrica aislada del retriever, igual que en el notebook final.
    aplicables = [
        v for k, v in ev.items()
        if k != "recall@5" and v is not None
    ]

    return {
        "id": caso["id"],
        "familia": caso["familia"],
        "pregunta": caso["pregunta"],
        "respuesta_esperada": caso["respuesta_esperada"],
        "respuesta_en_corpus": caso["respuesta_en_corpus"],
        "fuente_esperada": caso["fuente_esperada"],
        "herramienta_esperada": caso["herramienta_esperada"],
        "respuesta": _respuesta_dict(resultado),
        "trayectoria": _trayectoria(resultado),
        "evaluacion": ev,
        "acierto": bool(aplicables) and all(aplicables) and error is None,
        "coste_usd": _coste(resultado),
        "latencia_s": latencia,
        "n_llamadas": len(miax_s2.herramientas_usadas(resultado)),
        "error": error,
    }


def evaluar_holdout_profesor(ruta_jsonl="holdout.jsonl"):
    casos = leer_jsonl(ruta_jsonl)
    problemas = validar_holdout_profesor(casos)
    if problemas:
        raise ValueError("\n".join(problemas))

    registros = [_evaluar_una_holdout(c) for c in casos]

    filas = []
    for r in registros:
        respuesta = r["respuesta"] or {}
        filas.append({
            "id": r["id"],
            "familia": r["familia"],
            "en_corpus": r["respuesta_en_corpus"],
            "acierto": r["acierto"],
            "evidencia": r["evaluacion"]["evidencia"],
            "cifra": r["evaluacion"]["cifra"],
            "fuente": r["evaluacion"]["fuente"],
            "trayectoria": r["evaluacion"]["trayectoria"],
            "ausencia": r["evaluacion"]["ausencia"],
            "recall": r["evaluacion"]["recall@5"],
            "fuente_esperada": r["fuente_esperada"],
            "fuente_agente": respuesta.get("fuente"),
            "cifra_agente": respuesta.get("cifra"),
            "chunk_id": respuesta.get("chunk_id"),
            "respuesta_agente": respuesta.get("respuesta"),
            "coste_usd": r["coste_usd"],
            "latencia_s": r["latencia_s"],
            "llamadas": r["n_llamadas"],
            "error": r["error"],
        })

    return registros, pd.DataFrame(filas)

### Ejecutar las 10 preguntas del profesor

In [16]:
CARPETA_HOLDOUT = Path("resultados_holdout_profesor")
CARPETA_HOLDOUT.mkdir(exist_ok=True)

P_JSON = CARPETA_HOLDOUT / "holdout_registros.jsonl"
P_CSV = CARPETA_HOLDOUT / "holdout_tabla.csv"

# Por defecto se ejecuta el holdout al hacer Run All.
# Si vuelves a ejecutar el notebook en la MISMA carpeta y quieres evitar
# nuevas llamadas a Gemini, cambia a False.
EJECUTAR_HOLDOUT = True

if EJECUTAR_HOLDOUT:
    registros_holdout, tabla_holdout = evaluar_holdout_profesor(RUTA_HOLDOUT)

    with P_JSON.open("w", encoding="utf-8") as f:
        for r in registros_holdout:
            f.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    tabla_holdout.to_csv(P_CSV, index=False)
    print("Holdout ejecutado y guardado.")
else:
    if not (P_JSON.is_file() and P_CSV.is_file()):
        raise FileNotFoundError(
            "EJECUTAR_HOLDOUT=False pero no existen resultados guardados."
        )
    registros_holdout = leer_jsonl(P_JSON)
    tabla_holdout = pd.read_csv(P_CSV)
    print("Resultados previos del holdout cargados.")

display(
    tabla_holdout[
        [
            "id", "familia", "en_corpus", "acierto",
            "evidencia", "cifra", "fuente", "trayectoria",
            "fuente_esperada", "fuente_agente",
            "chunk_id", "coste_usd", "latencia_s", "llamadas", "error",
        ]
    ]
)

Holdout ejecutado y guardado.


,id,familia,en_corpus,acierto,evidencia,cifra,fuente,trayectoria,fuente_esperada,fuente_agente,chunk_id,coste_usd,latencia_s,llamadas,error
0,ho-001,extractiva,True,True,True,None,True,True,texto,texto,NVDA-2024-1A-0005,0.009375,8.674328,2,None
1,ho-002,extractiva,True,True,True,None,True,True,texto,texto,GOOGL-2025-7A-0000,0.005056,3.253840,2,None
2,ho-003,extractiva,True,True,True,None,True,True,texto,texto,AAPL-2025-7-0002,0.012712,9.029462,4,None
3,ho-004,numerica,True,True,None,True,True,True,xbrl,xbrl,None,0.003015,2.885564,2,None
4,ho-005,numerica,False,False,None,True,False,True,ninguna,texto,AMZN-2025-7-0019,0.023919,12.412359,6,None
5,ho-006,numerica,False,False,None,False,False,True,ninguna,texto,NVDA-2024-8-0006,0.010411,9.311470,4,None
6,ho-007,comparativa,True,True,True,True,True,True,ambas,ambas,AAPL-2025-7-0003,0.012104,10.187191,5,None
7,ho-008,comparativa,True,True,True,True,True,True,ambas,ambas,GOOGL-2025-7-0006,0.026156,11.883756,7,None
8,ho-009,comparativa,True,True,True,True,True,True,ambas,ambas,NVDA-2025-7-0015,0.013520,10.844761,5,None
9,ho-010,comparativa,True,True,True,True,True,True,ambas,ambas,MSFT-2025-7-0007,0.014818,10.749476,5,None


### Resultados del holdout y diagnóstico por pregunta

In [17]:
def resumen_holdout(tabla):
    return {
        "n": len(tabla),
        "aciertos": int(tabla.acierto.sum()),
        "accuracy": tabla.acierto.mean(),
        "recall@5": tabla.recall.dropna().mean(),
        "coste_medio_usd": tabla.coste_usd.mean(),
        "latencia_media_s": tabla.latencia_s.mean(),
        "llamadas_por_pregunta": tabla.llamadas.mean(),
    }

res_holdout = resumen_holdout(tabla_holdout)

resumen_holdout_df = pd.DataFrame([res_holdout])
display(
    resumen_holdout_df.style.format({
        "accuracy": "{:.1%}",
        "recall@5": "{:.1%}",
        "coste_medio_usd": "${:.4f}",
        "latencia_media_s": "{:.2f}",
        "llamadas_por_pregunta": "{:.2f}",
    })
)

por_familia_holdout = (
    tabla_holdout.groupby("familia")
    .agg(
        aciertos=("acierto", "sum"),
        n=("acierto", "size"),
        accuracy=("acierto", "mean"),
        recall_5=("recall", "mean"),
        coste_medio_usd=("coste_usd", "mean"),
        latencia_media_s=("latencia_s", "mean"),
        llamadas=("llamadas", "mean"),
    )
    .reset_index()
)
por_familia_holdout["resultado"] = (
    por_familia_holdout["aciertos"].astype(int).astype(str)
    + "/"
    + por_familia_holdout["n"].astype(int).astype(str)
)

display(
    por_familia_holdout[
        [
            "familia", "resultado", "accuracy", "recall_5",
            "coste_medio_usd", "latencia_media_s", "llamadas"
        ]
    ].style.format({
        "accuracy": "{:.1%}",
        "recall_5": "{:.1%}",
        "coste_medio_usd": "${:.4f}",
        "latencia_media_s": "{:.2f}",
        "llamadas": "{:.2f}",
    })
)

fallos_holdout = tabla_holdout.loc[
    ~tabla_holdout.acierto,
    [
        "id", "familia", "respuesta_agente",
        "evidencia", "cifra", "fuente", "trayectoria",
        "fuente_esperada", "fuente_agente", "chunk_id", "error"
    ],
]
print("Fallos del holdout:", len(fallos_holdout))
display(fallos_holdout)

,n,aciertos,accuracy,recall@5,coste_medio_usd,latencia_media_s,llamadas_por_pregunta
0,10,8,80.0%,57.1%,$0.0131,8.92,4.20


,familia,resultado,accuracy,recall_5,coste_medio_usd,latencia_media_s,llamadas
0,comparativa,4/4,100.0%,50.0%,$0.0166,10.92,5.50
1,extractiva,3/3,100.0%,66.7%,$0.0090,6.99,2.67
2,numerica,1/3,33.3%,nan%,$0.0124,8.20,4.00


Fallos del holdout: 2


,id,familia,respuesta_agente,evidencia,cifra,fuente,trayectoria,fuente_esperada,fuente_agente,chunk_id,error
4,ho-005,numerica,Amazon no reporta la cifra de beneficio bruto ...,None,True,False,True,ninguna,texto,AMZN-2025-7-0019,None
5,ho-006,numerica,En el ejercicio fiscal 2023 (año fiscal finali...,None,False,False,True,ninguna,texto,NVDA-2024-8-0006,None


## 7. Comparación · nuestro golden vs holdout del profesor

La fila **Golden propio** usa la ejecución final congelada que ya habíamos
cerrado antes de conocer el holdout. No se recalcula ni se ajusta después de
ver estas 10 preguntas.

Las poblaciones no son idénticas: nuestro golden tiene 20 preguntas
(7 numéricas, 7 extractivas y 6 comparativas); el holdout tiene 10
(3 numéricas, 3 extractivas y 4 comparativas), incluidos dos casos
deliberadamente fuera de corpus. Por eso la comparación es descriptiva, no una
estimación estadística de superioridad.

In [18]:
# Métricas de la ejecución final congelada de nuestro golden.
PROPIO_FINAL = {
    "conjunto": "Golden propio (20)",
    "n": 20,
    "aciertos": 20,
    "accuracy": 1.00,
    "recall@5": 8 / 13,        # 61,5 %
    "coste_medio_usd": 0.0122,
    "latencia_media_s": 8.73,
    "llamadas_por_pregunta": 4.05,
}

PROFESOR_HOLDOUT = {
    "conjunto": "Holdout profesor (10)",
    "n": res_holdout["n"],
    "aciertos": res_holdout["aciertos"],
    "accuracy": res_holdout["accuracy"],
    "recall@5": res_holdout["recall@5"],
    "coste_medio_usd": res_holdout["coste_medio_usd"],
    "latencia_media_s": res_holdout["latencia_media_s"],
    "llamadas_por_pregunta": res_holdout["llamadas_por_pregunta"],
}

comparacion_prop_vs_holdout = pd.DataFrame(
    [PROPIO_FINAL, PROFESOR_HOLDOUT]
)

display(
    comparacion_prop_vs_holdout.style.format({
        "accuracy": "{:.1%}",
        "recall@5": "{:.1%}",
        "coste_medio_usd": "${:.4f}",
        "latencia_media_s": "{:.2f}",
        "llamadas_por_pregunta": "{:.2f}",
    })
)

# Comparación de accuracy por familia.
familias_propias = pd.DataFrame([
    {"conjunto": "Golden propio", "familia": "numerica", "aciertos": 7, "n": 7},
    {"conjunto": "Golden propio", "familia": "extractiva", "aciertos": 7, "n": 7},
    {"conjunto": "Golden propio", "familia": "comparativa", "aciertos": 6, "n": 6},
])

familias_holdout_comp = por_familia_holdout[
    ["familia", "aciertos", "n"]
].copy()
familias_holdout_comp.insert(0, "conjunto", "Holdout profesor")

comparacion_familias = pd.concat(
    [familias_propias, familias_holdout_comp],
    ignore_index=True,
)
comparacion_familias["accuracy"] = (
    comparacion_familias["aciertos"] / comparacion_familias["n"]
)
comparacion_familias["resultado"] = (
    comparacion_familias["aciertos"].astype(int).astype(str)
    + "/"
    + comparacion_familias["n"].astype(int).astype(str)
)

display(
    comparacion_familias.pivot(
        index="conjunto",
        columns="familia",
        values="accuracy",
    ).style.format("{:.1%}")
)

,conjunto,n,aciertos,accuracy,recall@5,coste_medio_usd,latencia_media_s,llamadas_por_pregunta
0,Golden propio (20),20,20,100.0%,61.5%,$0.0122,8.73,4.05
1,Holdout profesor (10),10,8,80.0%,57.1%,$0.0131,8.92,4.20


familia,comparativa,extractiva,numerica
conjunto,,,
Golden propio,100.0%,100.0%,100.0%
Holdout profesor,100.0%,100.0%,33.3%


## 8. Guardar y descargar resultados

In [19]:
# Guardamos todas las tablas útiles.
resumen_holdout_df.to_csv(
    CARPETA_HOLDOUT / "resumen_holdout.csv",
    index=False,
)
por_familia_holdout.to_csv(
    CARPETA_HOLDOUT / "holdout_por_familia.csv",
    index=False,
)
fallos_holdout.to_csv(
    CARPETA_HOLDOUT / "holdout_fallos.csv",
    index=False,
)
comparacion_prop_vs_holdout.to_csv(
    CARPETA_HOLDOUT / "comparacion_golden_vs_holdout.csv",
    index=False,
)
comparacion_familias.to_csv(
    CARPETA_HOLDOUT / "comparacion_por_familia.csv",
    index=False,
)

import shutil as _shutil

zip_path = _shutil.make_archive(
    "resultados_holdout_profesor",
    "zip",
    root_dir=CARPETA_HOLDOUT,
)
print("ZIP generado:", zip_path)

try:
    from google.colab import files
    files.download("resultados_holdout_profesor.zip")
except ImportError:
    print("Fuera de Colab: el ZIP queda en", zip_path)

ZIP generado: /content/MIAX_2026/Practica_Agente_RAG/resultados_holdout_profesor.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>